# Feature Engineering

Tier A/B feature builders (Decision #1), target definition/transforms (Decisions #2, #3), failed-job exclusion (Decision #4), embedding dimensionality reduction (Decision #7).

See the conceptualization plan and EXPERIMENT_TRACKER.md for full context.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, features, metrics, plotting, models, roofline


## Load data

Same `SAMPLE_SIZE` dev-sample pattern as notebook 01 — see `EXPERIMENT_TRACKER.md` for when this needs to step up toward the full dataset.

In [ ]:
FDATA_DIR = "../data/raw/fdata"
PM100_PATH = "../data/raw/pm100/pm100_job_table.parquet"
SAMPLE_SIZE = 5000

# Same rationale as notebook 01: with all 38 months present, do NOT load
# every file in full before sampling — the embedding column alone would
# need ~100GB+ for all ~26M rows. Load a handful of files, sample from
# that smaller pool instead.
N_FILES_FOR_SAMPLE = 5

import glob
fdata_files = sorted(glob.glob(f"{FDATA_DIR}/*.parquet"))

rng = np.random.RandomState(0)
sampled_files = sorted(rng.choice(fdata_files, size=min(N_FILES_FOR_SAMPLE, len(fdata_files)), replace=False))
print(f"Loading {len(sampled_files)} of {len(fdata_files)} files: {sampled_files}")

fdata = pd.concat([pd.read_parquet(f) for f in sampled_files], ignore_index=True)
pm100 = pd.read_parquet(PM100_PATH)

if SAMPLE_SIZE is not None:
    fdata = fdata.sample(n=min(SAMPLE_SIZE, len(fdata)), random_state=0).reset_index(drop=True)
    pm100 = pm100.sample(n=min(SAMPLE_SIZE, len(pm100)), random_state=0).reset_index(drop=True)

FDATA_DATETIME_COLS = ["adt", "qdt", "schedsdt", "deldt", "sdt", "edt"]
for c in FDATA_DATETIME_COLS:
    fdata[c] = pd.to_datetime(fdata[c], errors="coerce")
pm100["submit_time"] = pd.to_datetime(pm100["submit_time"], unit="s", errors="coerce")
pm100["start_time"] = pd.to_datetime(pm100["start_time"], unit="s", errors="coerce")
pm100["end_time"] = pd.to_datetime(pm100["end_time"], unit="s", errors="coerce")

print("fdata:", fdata.shape, " pm100:", pm100.shape)


## Step 1: Exclude failed/cancelled/killed jobs (Decision #4)

Their duration/power are artifacts of failure, not real workload behavior. F-DATA's `exit state` is a clean completed/failed binary; PM100's `job_state` has 6 values, only `COMPLETED` reflects normal execution.

In [3]:
print("F-DATA exit state breakdown:")
print(fdata["exit state"].value_counts())
print("\nPM100 job_state breakdown:")
print(pm100["job_state"].value_counts())

fdata = features.filter_completed_jobs(fdata, "fdata")
pm100 = features.filter_completed_jobs(pm100, "pm100")


F-DATA exit state breakdown:
exit state
completed    4743
failed        257
Name: count, dtype: int64

PM100 job_state breakdown:
job_state
COMPLETED        3952
FAILED            627
CANCELLED         208
TIMEOUT           196
OUT_OF_MEMORY      16
NODE_FAIL           1
Name: count, dtype: int64
[fdata] excluded 5.1% of jobs as non-completed (257 / 5000)
[pm100] excluded 21.0% of jobs as non-completed (1048 / 5000)


## Step 2: Tier A/B feature matrices (Decision #1)

Re-running the leakage sanity check from notebook 01 on the post-exclusion data — this should still pass, but re-verifying after every transformation step is the point of Decision #19.

In [4]:
fdata_tier_a = features.build_tier_a_features(fdata, "fdata")
fdata_tier_b = features.build_tier_b_features(fdata, "fdata")
pm100_tier_a = features.build_tier_a_features(pm100, "pm100")
pm100_tier_b = features.build_tier_b_features(pm100, "pm100")

for name, cols, tier, dataset in [
    ("F-DATA Tier A", fdata_tier_a.columns, "A", "fdata"),
    ("F-DATA Tier B", fdata_tier_b.columns, "B", "fdata"),
    ("PM100 Tier A", pm100_tier_a.columns, "A", "pm100"),
    ("PM100 Tier B", pm100_tier_b.columns, "B", "pm100"),
]:
    features.assert_no_tier_leakage(list(cols), tier, dataset)
    print(f"{name}: {len(cols)} columns, OK")


F-DATA Tier A: 13 columns, OK
F-DATA Tier B: 30 columns, OK
PM100 Tier A: 18 columns, OK
PM100 Tier B: 15 columns, OK


## Step 3: Embedding dimensionality reduction (Decision #7, Option C)

F-DATA only — PM100 has no job-name embedding field. Rather than picking an arbitrary component count, first check cumulative explained variance and pick the smallest number reaching 90% — then use that same PCA-reduced representation everywhere (tree models and DL alike, per Option C — the plan's original allowance for DL to use the full raw embedding is intentionally not exercised, to keep the memory profile safe unconditionally).

In [ ]:
pca_variance_check = features.compute_embedding_explained_variance(fdata, max_components=100)

import numpy as np
cumulative = np.cumsum(pca_variance_check.explained_variance_ratio_)
for threshold in [0.80, 0.90, 0.95, 0.99]:
    n = features.n_components_for_variance(pca_variance_check, threshold)
    print(f"{threshold:.0%} variance -> {n} components")

plt.figure(figsize=(6, 4))
plt.plot(range(1, len(cumulative) + 1), cumulative)
plt.axhline(0.90, color="r", linestyle="--", linewidth=1, label="90% threshold")
plt.xlabel("Number of PCA components")
plt.ylabel("Cumulative explained variance")
plt.title("F-DATA embedding: cumulative explained variance")
plt.legend()
plt.show()

N_EMBEDDING_COMPONENTS = features.n_components_for_variance(pca_variance_check, threshold=0.90)
print(f"\nUsing N_EMBEDDING_COMPONENTS={N_EMBEDDING_COMPONENTS} (90% explained variance) going forward")


In [ ]:
fdata_emb_pca = features.reduce_fdata_embedding(fdata, n_components=N_EMBEDDING_COMPONENTS)
print(fdata_emb_pca.shape)
fdata_emb_pca.describe().T


## Step 4: Historical rolling-stat features (Tier A)

Each user's mean `duration` over their *previous* jobs only (never the current job's own outcome — computed via `shift(1)` before the rolling window, per Decision #1's leakage discipline). With only 1 of F-DATA's 38 months present and a 5000-row dev sample, expect these to be sparse/noisy — this is expected, not a bug (see EXPERIMENT_TRACKER.md).

In [6]:
fdata_with_rolling = features.add_user_rolling_stat(fdata, "fdata", "duration", window=5)
pm100_with_rolling = features.add_user_rolling_stat(pm100, "pm100", "run_time", window=5)

n_fdata_available = fdata_with_rolling["duration_user_rolling_mean"].notna().sum()
n_pm100_available = pm100_with_rolling["run_time_user_rolling_mean"].notna().sum()
print(f"F-DATA: rolling stat available for {n_fdata_available}/{len(fdata_with_rolling)} jobs")
print(f"PM100: rolling stat available for {n_pm100_available}/{len(pm100_with_rolling)} jobs")

fdata_with_rolling[["usr", "adt", "duration", "duration_user_rolling_mean"]].head(10)


F-DATA: rolling stat available for 4586/4743 jobs
PM100: rolling stat available for 3717/3952 jobs


,usr,adt,duration,duration_user_rolling_mean
1,usr_1226,2022-01-01 23:33:34+09:00,446.0,1116.4
2,usr_1365,2022-01-31 20:57:51+09:00,209.0,128.2
4,usr_623,2022-01-13 14:18:12+09:00,8120.0,904.0
5,usr_896,2022-01-26 13:10:04+09:00,108.0,124.0
6,usr_1012,2022-01-23 16:19:22+09:00,1.0,1.2
7,usr_617,2022-01-06 10:40:53+09:00,189.0,180.0
8,usr_1226,2022-01-01 20:51:02+09:00,133.0,1696.2
9,usr_1122,2022-01-13 14:15:28+09:00,529.0,243.0
10,usr_147,2022-01-26 19:01:16+09:00,25.0,65.2
11,usr_1226,2022-01-01 10:39:25+09:00,1164.0,887.8


## Step 5: Target transforms (Decision #3)

log1p on all three targets per dataset (execution time, memory, power) — training happens in log-space, metrics get reported in both log-space and back-transformed real units. Sanity-checking the round-trip here (Decision #19) rather than assuming it.

In [7]:
for name, targets, df in [("F-DATA", features.FDATA_TARGETS, fdata), ("PM100", features.PM100_TARGETS, pm100)]:
    for target_name, col in targets.items():
        raw = df[col].dropna().to_numpy()
        if len(raw) and hasattr(raw[0], "__len__"):
            # PM100 power: array-valued, reduce to per-job mean first (see notebook 01)
            raw = np.array([np.mean(a) for a in raw])
        raw = raw[raw >= 0].astype(float)
        transformed = features.transform_target(raw)
        recovered = features.inverse_transform_target(transformed)
        metrics.expm1_round_trip_check(raw)
        print(f"{name} {target_name} ({col}): round-trip OK, n={len(raw)}, "
              f"raw range [{raw.min():.2f}, {raw.max():.2f}], "
              f"log1p range [{transformed.min():.2f}, {transformed.max():.2f}]")


F-DATA execution_time (duration): round-trip OK, n=4743, raw range [1.00, 234313.00], log1p range [0.69, 12.36]
F-DATA memory (mmszu): round-trip OK, n=4743, raw range [24444928.00, 29955457024.00], log1p range [17.01, 24.12]
F-DATA power (avgpcon): round-trip OK, n=4743, raw range [39.65, 229976.49], log1p range [3.71, 12.35]
PM100 execution_time (run_time): round-trip OK, n=3952, raw range [0.00, 86328.00], log1p range [0.00, 11.37]
PM100 memory (mem_alloc): round-trip OK, n=3952, raw range [1.00, 60800.00], log1p range [0.69, 11.02]
PM100 power (node_power_consumption): round-trip OK, n=3952, raw range [480.00, 173911.93], log1p range [6.18, 12.07]


## Summary

This notebook validates the feature-engineering pipeline end to end on the dev sample: exclusion, Tier A/B separation, embedding reduction, rolling-stat features, and target transforms all pass their sanity checks. Nothing here is yet the "final" feature matrix used for training — that assembly happens in notebooks 05-07, reusing these same `src/features.py` functions against the full dataset once available.